# **Mount the Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Import Dependencies**

In [2]:
!pip -q install shapely pyogrio rtree

In [3]:
import geemap
import ee
import geopandas as gpd
import pandas as pd
import numpy as np
import seaborn as sns
import random
from shapely.geometry import Point
from pyproj import CRS
from shapely.prepared import prep
import os

In [5]:
ee.Authenticate()
ee.Initialize(project='ee-ranitsundarchatterjee')

In [6]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# **Prepare & Visualize the Data**

## **Prepare**

In [7]:
Point_data = ee.FeatureCollection('projects/ee-ranitsundarchatterjee/assets/Crop_Classification/Data_Points')
Polygon_data = ee.FeatureCollection('projects/ee-ranitsundarchatterjee/assets/Crop_Classification/Data_polygons')

# =====================================================================
# CONVERT GEE DATA TO GEOPANDAS
# =====================================================================
print("Downloading Earth Engine assets...")
gdf_points = geemap.ee_to_gdf(Point_data)
gdf_polygons = geemap.ee_to_gdf(Polygon_data)

# Project to local UTM zone to calculate accurate meter distances
utm_crs = gdf_polygons.estimate_utm_crs()
gdf_points_proj = gdf_points.to_crs(utm_crs)
gdf_polygons_proj = gdf_polygons.to_crs(utm_crs)

In [ ]:
Final till date: 27/07/26 (08:46PM)
# =====================================================================
# 5. GRID GENERATION AND DATA INHERITANCE
# =====================================================================
print("Preserving 100% of original points and applying T&C for new additions...")

GRID_SPACING = 15       # Target spacing grid in meters
BOUNDARY_BUFFER = -5     # Inward shrink for safe new point placement
MIN_DISTANCE = 15        # T&C: Minimum distance between ANY two points in meters

final_rows = []

# --- PHASE 1: KEEP ALL ORIGINAL POINTS UNCONDITIONALLY ---
for _, orig_row in gdf_points_proj.iterrows():
    # Remove index helper if it exists, otherwise keep as is
    clean_orig_row = orig_row.drop(['index_right'], errors='ignore')
    final_rows.append(clean_orig_row)

print(f"Phase 1 Complete: Added all {len(final_rows)} original points to final collection.")

# --- PHASE 2: GENERATE AND ADD NEW POINTS BASED ON T&C ---
# Spatial join to link points with polygon metadata for candidate attribute copying
joined = gpd.sjoin(gdf_points_proj, gdf_polygons_proj, how='inner', predicate='within')

for idx, poly_feature in gdf_polygons_proj.iterrows():
    polygon_geom = poly_feature.geometry

    # Find existing points belonging to this polygon to inherit metadata
    poly_existing_points = joined[joined['index_right'] == idx]

    # If no original point exists in this polygon, we can't inherit attributes
    if poly_existing_points.empty:
        continue

    # Create the internal safe zone for new point generation (T&C: edge buffer)
    safe_generation_zone = polygon_geom.buffer(BOUNDARY_BUFFER)
    if safe_generation_zone.is_empty or not safe_generation_zone.is_valid:
        continue

    # Track points inside this polygon to enforce the 20m rule
    # Starts with ALL original points that fall within this polygon
    accepted_geometries = list(poly_existing_points.geometry)

    # Build bounding box grid
    min_x, min_y, max_x, max_y = safe_generation_zone.bounds
    x_coords = np.arange(min_x, max_x, GRID_SPACING)
    y_coords = np.arange(min_y, max_y, GRID_SPACING)

    grid_points = [Point(x, y) for x in x_coords for y in y_coords]

    # Check candidates against terms & conditions
    for candidate in grid_points:
        # T&C 1: Must be inside the buffered safe zone
        if safe_generation_zone.contains(candidate):

            # T&C 2: Must be >= 20m from ALL accepted points (original + new)
            too_close = False
            for accepted_geom in accepted_geometries:
                if candidate.distance(accepted_geom) < MIN_DISTANCE:
                    too_close = True
                    break

            # If it satisfies all conditions, add it!
            if not too_close:
                # Find nearest original point in this polygon to copy metadata
                closest_point_row = min(
                    poly_existing_points.iterrows(),
                    key=lambda r: candidate.distance(r[1].geometry)
                )[1]

                # Create a copy of the closest point's attributes, excluding geometry and 'index_right'
                # Drop any _right suffixed columns which originate from the polygon feature
                attributes_to_copy = closest_point_row.drop(
                    ['geometry', 'index_right'] + [col for col in closest_point_row.index if col.endswith('_right')],
                    errors='ignore'
                ).copy()

                # Define the renaming map for _left suffixed columns
                rename_map_for_new_points = {
                    'collection_left': 'collection',
                    'crop_left': 'crop',
                    'district_left': 'district',
                    'fid__left': 'fid_',
                    'harvest_left': 'harvest',
                    'id_left': 'id',
                    'latitude_left': 'latitude',
                    'longitude_left': 'longitude',
                    'pheno_stag_left': 'pheno_stag',
                    'season_left': 'season',
                    'sowing_left': 'sowing',
                    'state_left': 'state'
                }

                # Apply renaming to the Series index (which are the attribute names)
                # Only rename if the key exists in the Series' index to avoid errors
                attributes_to_copy = attributes_to_copy.rename(
                    index={k: v for k, v in rename_map_for_new_points.items() if k in attributes_to_copy.index}
                )

                # Now, construct the final new_row with the new geometry and cleaned attributes
                new_row_dict = attributes_to_copy.to_dict()
                new_row_dict['geometry'] = candidate

                # Create a new Series for the new point
                clean_new_row = pd.Series(new_row_dict)

                # Append to output dataset
                final_rows.append(clean_new_row)

                # Register new point so future candidates maintain 20m distance from it
                accepted_geometries.append(candidate)

print(f"Phase 2 Complete: Total points now = {len(final_rows)}")

Preserving 100% of original points and applying T&C for new additions...
Phase 1 Complete: Added all 45616 original points to final collection.
Phase 2 Complete: Total points now = 75565


In [9]:
# Convert the list of final rows to a GeoDataFrame
gdf_final_points_proj = gpd.GeoDataFrame(final_rows, crs=utm_crs)

# Convert the GeoDataFrame to WGS84 (EPSG:4326)
gdf_final_points_wgs84 = gdf_final_points_proj.to_crs(epsg=4326)

# 1. Drop all columns ending with '_right'
right_cols_to_drop = [col for col in gdf_final_points_wgs84.columns if col.endswith('_right')]
gdf_final_points_wgs84 = gdf_final_points_wgs84.drop(columns=right_cols_to_drop)

# Define rename mapping for '_left' suffixed columns
rename_dict = {
    'collection_left': 'collection',
    'crop_left': 'crop',
    'district_left': 'district',
    'fid__left': 'fid_',
    'harvest_left': 'harvest',
    'id_left': 'id',
    'latitude_left': 'latitude',
    'longitude_left': 'longitude',
    'pheno_stag_left': 'pheno_stag',
    'season_left': 'season',
    'sowing_left': 'sowing',
    'state_left': 'state'
}

# Identify original columns that will be duplicated by renaming '_left' columns
cols_to_drop_before_rename = []
for old_name, new_name in rename_dict.items():
    if old_name in gdf_final_points_wgs84.columns and new_name in gdf_final_points_wgs84.columns:
        cols_to_drop_before_rename.append(new_name)

# Drop identified original columns to prevent duplication
if cols_to_drop_before_rename:
    gdf_final_points_wgs84 = gdf_final_points_wgs84.drop(columns=cols_to_drop_before_rename)

# Rename '_left' suffixed columns back to clean names
# Only apply rename for columns that actually exist
existing_rename_dict = {k: v for k, v in rename_dict.items() if k in gdf_final_points_wgs84.columns}
gdf_final_points_wgs84 = gdf_final_points_wgs84.rename(columns=existing_rename_dict)


# 2. Update latitude and longitude columns to match the newly generated point geometries
gdf_final_points_wgs84['longitude'] = gdf_final_points_wgs84.geometry.x
gdf_final_points_wgs84['latitude'] = gdf_final_points_wgs84.geometry.y

# 3. Handle 'NA' values if present
if 'harvest' in gdf_final_points_wgs84.columns:
    gdf_final_points_wgs84['harvest'] = gdf_final_points_wgs84['harvest'].replace('NA', np.nan)

# 4. Define final clean column order
final_column_order = [
    'collection', 'crop', 'district', 'fid_', 'harvest', 'id',
    'latitude', 'longitude', 'pheno_stag', 'season', 'sowing',
    'state', 'area_ha', 'geometry'
]

# Reorder columns (safely checking for existence)
existing_order = [col for col in final_column_order if col in gdf_final_points_wgs84.columns]
gdf_final_points_wgs84 = gdf_final_points_wgs84[existing_order]

print("Updated Columns:")
print(gdf_final_points_wgs84.columns)

Updated Columns:
Index(['collection', 'crop', 'district', 'fid_', 'harvest', 'id', 'latitude',
       'longitude', 'pheno_stag', 'season', 'sowing', 'state', 'area_ha',
       'geometry'],
      dtype='object')


## **Add target Column 'Label'**

In [10]:
# 1. Copy the values from the 'crop' column into a new 'Label' column
gdf_final_points_wgs84['Label'] = gdf_final_points_wgs84['crop']

# 2. Get all column names except 'Label'
existing_cols = [col for col in gdf_final_points_wgs84.columns if col != 'Label']

# 3. Reorder so 'Label' is at the very end
gdf_final_points_wgs84 = gdf_final_points_wgs84[existing_cols + ['Label']]

print("Updated Columns:")
gdf_final_points_wgs84.columns

Updated Columns:


Index(['collection', 'crop', 'district', 'fid_', 'harvest', 'id', 'latitude',
       'longitude', 'pheno_stag', 'season', 'sowing', 'state', 'area_ha',
       'geometry', 'Label'],
      dtype='object')

## **Visualization**

In [ ]:
print(f"Original number of points: {len(gdf_final_points_wgs84)}")

# Sample a subset of points for visualization to avoid payload size limit
# Adjust 'n' as needed, or remove sampling if you export to an EE asset
sampled_points = gdf_final_points_wgs84.sample(n=50, random_state=42)
print(f"Displaying a sample of {len(sampled_points)} points on the map.")

point_ee = geemap.gdf_to_ee(sampled_points)
Map.addLayer(point_ee, {}, "Points")
Map.centerObject(point_ee, 10)

Original number of points: 75565
Displaying a sample of 50 points on the map.


## **Download the Data**

In [11]:
# Define output directory and file path
output_folder = "/content/drive/MyDrive/Crop_Classification/Data/Shapefiles/Updated_Points/"
os.makedirs(output_folder, exist_ok=True)

# Save GeoDataFrame as ESRI Shapefile
shp_path = os.path.join(output_folder, "Extra_Added_Points.shp")
gdf_final_points_wgs84.to_file(shp_path, driver="ESRI Shapefile")
print(f"New points saved to {shp_path}")

New points saved to /content/drive/MyDrive/Crop_Classification/Data/Shapefiles/Updated_Points/Extra_Added_Points_V2.shp


In [ ]:
# Define output directory and CSV file path
output_folder = "/content/drive/MyDrive/Crop_Classification/Data/Shapefiles/Updated_Points/"
os.makedirs(output_folder, exist_ok=True)

csv_path = os.path.join(output_folder, "Extra_Added_Points.csv")

# Create a copy so we don't modify the original GeoDataFrame
df_csv = gdf_final_points_wgs84.copy()

# Extract WGS84 Latitude and Longitude coordinates into standalone columns
df_csv['longitude'] = df_csv.geometry.x
df_csv['latitude'] = df_csv.geometry.y

# Drop geometry object column for standard CSV export (or keep if desired)
df_csv_export = df_csv.drop(columns=['geometry'], errors='ignore')

# Save to CSV
df_csv_export.to_csv(csv_path, index=False)

print(f"New points successfully saved to CSV at: {csv_path}")

New points successfully saved to CSV at: /content/drive/MyDrive/Crop_Classification/Data/Shapefiles/Updated_Points/Extra_Added_Points.csv
